In [46]:
import yaml
import os
import polars as pl
from datetime import datetime ,date

with open("../config/config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

data_dir = config["data_dir"]
print(data_dir  )
if os.path.exists(data_dir) and os.path.isdir(data_dir):
    list_of_files = os.listdir(data_dir)
else:
    list_of_files = []
print(list_of_files)

E:\smallcaps_data
['AIXI.parquet', 'APUS.parquet', 'BAOS.parquet', 'BEAT.parquet', 'BIAF.parquet', 'CAN.parquet', 'CDTG.parquet', 'CHAI.parquet', 'CHPT.parquet', 'CISS.parquet', 'COOT.parquet', 'CYAB.parquet', 'FBGL.parquet', 'FLYE.parquet', 'GCDT.parquet', 'GIPR.parquet', 'GRI.parquet', 'GRML.parquet', 'IMCC.parquet', 'IMRN.parquet', 'LIDR.parquet', 'logs', 'MEDS.parquet', 'NCPL.parquet', 'NCRA.parquet', 'NEOV.parquet', 'OFAL.parquet', 'OLB.parquet', 'OLOX.parquet', 'output', 'PDSB.parquet', 'PFSA.parquet', 'SBFM.parquet', 'SDEV.parquet', 'SKYQ.parquet', 'SOAR.parquet', 'SPHL.parquet', 'STAK.parquet', 'TNMG.parquet', 'TOPP.parquet', 'TRUG.parquet', 'USDE.parquet', 'VEEA.parquet', 'VRA.parquet', 'VRME.parquet', 'XHLD.parquet', 'XPON.parquet', 'XTNT.parquet', '_catalog.parquet']


In [47]:
# list_of_files = [f for f in list_of_files if f.endswith(".parquet")]
list_of_tickers = [f for f in list_of_files if not f.startswith("_catalog") and f.endswith(".parquet")]
catalog = [f for f in list_of_files if f.startswith("_catalog") and f.endswith(".parquet")][0]
# parquet_path = os.path.join(data_dir, f"{ticker}.parquet")
catalog_path = os.path.join(data_dir, f"{catalog}")

df_catalog = pl.read_parquet(catalog_path)['ticker','end'].sort('ticker')
df_catalog = df_catalog.with_columns(
    pl.col('end').dt.date().alias('date')
)
df_catalog


ticker,end,date
str,"datetime[μs, UTC]",date
"""AIXI""",2026-08-07 23:59:00 UTC,2026-08-07
"""AIXI""",2026-08-28 23:59:00 UTC,2026-08-28
"""APUS""",2026-09-04 20:32:00 UTC,2026-09-04
"""APUS""",2026-09-24 23:59:00 UTC,2026-09-24
"""APUS""",2026-10-07 23:59:00 UTC,2026-10-07
…,…,…
"""XHLD""",2026-10-06 23:59:00 UTC,2026-10-06
"""XPON""",2026-08-04 20:05:00 UTC,2026-08-04
"""XPON""",2026-08-24 23:59:00 UTC,2026-08-24


In [48]:
df_backtest_runs = (
    df_catalog
    .select("ticker", "date")
    .unique()
    .sort(["ticker", "date"])
)
df_backtest_runs


ticker,date
str,date
"""AIXI""",2026-08-07
"""AIXI""",2026-08-28
"""APUS""",2026-09-04
"""APUS""",2026-09-24
"""APUS""",2026-10-07
…,…
"""XHLD""",2026-10-06
"""XPON""",2026-08-04
"""XPON""",2026-08-24


In [49]:
tickers_days = {
    row["ticker"]: row["days"]
    for row in (
        df_catalog
        .select(
            "ticker",
            pl.col("end").dt.strftime("%Y-%m-%d").alias("date"),
        )
        .group_by("ticker")
        .agg(
            pl.col("date")
            .unique()
            .sort(descending=True)
            .alias("days")
        )
        .sort("ticker")
        .to_dicts()
    )
}

tickers_days

{'AIXI': ['2026-08-28', '2026-08-07'],
 'APUS': ['2026-10-07', '2026-09-24', '2026-09-04'],
 'BAOS': ['2026-09-04', '2026-08-14'],
 'BEAT': ['2026-10-05', '2026-09-15'],
 'BIAF': ['2026-09-04', '2026-08-14'],
 'CAN': ['2026-08-21', '2026-07-31'],
 'CDTG': ['2026-09-04', '2026-08-14'],
 'CHAI': ['2026-08-28', '2026-08-07'],
 'CHPT': ['2026-09-03', '2026-08-14'],
 'CISS': ['2026-08-28', '2026-08-07'],
 'COOT': ['2026-08-31', '2026-08-11'],
 'CYAB': ['2026-08-28', '2026-08-07'],
 'FBGL': ['2026-09-22', '2026-09-02'],
 'FLYE': ['2026-10-08', '2026-09-18', '2026-09-01', '2026-08-12'],
 'GCDT': ['2026-09-23', '2026-09-03'],
 'GIPR': ['2026-08-25', '2026-08-24', '2026-08-04'],
 'GRI': ['2026-09-04', '2026-08-14'],
 'GRML': ['2026-09-25', '2026-09-04'],
 'IMCC': ['2026-09-22', '2026-09-02'],
 'IMRN': ['2026-09-04', '2026-08-14'],
 'LIDR': ['2026-09-01', '2026-08-12'],
 'MEDS': ['2026-10-08', '2026-09-18'],
 'NCPL': ['2026-09-21', '2026-09-01'],
 'NCRA': ['2026-08-31', '2026-08-11'],
 'NEOV': [

In [50]:

columns = ["datetime", "ticker", "open", "high", "low", "close", "volume"]
frames = []

for ticker, days in tickers_days.items():
    parquet_path = os.path.join(data_dir, f"{ticker}.parquet")

    if not os.path.isfile(parquet_path):
        raise FileNotFoundError(f"No existe el paquete Parquet para {ticker}: {parquet_path}")

    requested_days = [date.fromisoformat(day) for day in days]

    ticker_df = (
        pl.read_parquet(parquet_path)
        .filter(pl.col("datetime").dt.date().is_in(requested_days))
        .with_columns(pl.lit(ticker).alias("ticker"))
        .select(columns)
    )

    frames.append(ticker_df)

df = pl.concat(frames).sort(["ticker", "datetime"])
df

datetime,ticker,open,high,low,close,volume
"datetime[μs, UTC]",str,f64,f64,f64,f64,i64
2026-08-07 08:00:00 UTC,"""AIXI""",1.08,1.134,1.01,1.01,5617
2026-08-07 08:01:00 UTC,"""AIXI""",1.07,1.07,1.07,1.07,753
2026-08-07 08:02:00 UTC,"""AIXI""",1.05,1.05,1.05,1.05,142
2026-08-07 08:04:00 UTC,"""AIXI""",1.03,1.03,1.03,1.03,138
2026-08-07 08:06:00 UTC,"""AIXI""",1.0207,1.0207,1.0207,1.0207,974
…,…,…,…,…,…,…
2026-08-28 23:48:00 UTC,"""XTNT""",0.371,0.371,0.3666,0.3666,3638
2026-08-28 23:49:00 UTC,"""XTNT""",0.3709,0.371,0.3709,0.3709,5540
2026-08-28 23:57:00 UTC,"""XTNT""",0.3655,0.3699,0.3655,0.3699,703


In [51]:
GROUPS = ["ticker", "trading_day"]


def daily_vwap():
    return [
        (
            (pl.col("price") * pl.col("volume")).cum_sum().over(GROUPS)
            / pl.col("volume").cum_sum().over(GROUPS)
        ).alias("vwap_daily")
    ]


def atr(period=14):
    previous_close = pl.col("close").shift(1).over(GROUPS)

    true_range = pl.max_horizontal(
        pl.col("high") - pl.col("low"),
        (pl.col("high") - previous_close).abs(),
        (pl.col("low") - previous_close).abs(),
    )

    return [
        true_range
        .rolling_mean(window_size=period, min_samples=period)
        .over(GROUPS)
        .alias(f"atr_{period}")
    ]


def bollinger_bands(period=20, std_dev=2.0):
    middle = pl.col("close").rolling_mean(period, min_samples=period).over(GROUPS)
    deviation = (
        pl.col("close")
        .rolling_std(period, min_samples=period, ddof=0)
        .over(GROUPS)
    )

    return [
        middle.alias(f"bb_middle_{period}"),
        (middle + deviation * std_dev).alias(f"bb_upper_{period}"),
        (middle - deviation * std_dev).alias(f"bb_lower_{period}"),
    ]


def rsi(period=14):
    change = pl.col("close").diff().over(GROUPS)
    gain = pl.when(change > 0).then(change).otherwise(0.0)
    loss = pl.when(change < 0).then(-change).otherwise(0.0)

    average_gain = (
        gain.ewm_mean(alpha=1 / period, adjust=False, min_samples=period)
        .over(GROUPS)
    )
    average_loss = (
        loss.ewm_mean(alpha=1 / period, adjust=False, min_samples=period)
        .over(GROUPS)
    )

    return [
        pl.when(average_loss == 0)
        .then(100.0)
        .when(average_gain == 0)
        .then(0.0)
        .otherwise(100.0 - (100.0 / (1.0 + average_gain / average_loss)))
        .alias(f"rsi_{period}")
    ]


def donchian_channels(period=20):
    upper = pl.col("high").rolling_max(period, min_samples=period).over(GROUPS)
    lower = pl.col("low").rolling_min(period, min_samples=period).over(GROUPS)

    return [
        upper.alias(f"donchian_upper_{period}"),
        lower.alias(f"donchian_lower_{period}"),
        ((upper + lower) / 2).alias(f"donchian_middle_{period}"),
    ]


def momentum(period=10):
    return [
        (
            pl.col("close")
            - pl.col("close").shift(period).over(GROUPS)
        ).alias(f"momentum_{period}")
    ]


def add_indicators(frame, indicator_functions):
    expressions = [
        expression
        for indicator_function in indicator_functions
        for expression in indicator_function()
    ]
    return frame.with_columns(expressions)


df_to_plot = (
    df
    .with_columns(
        pl.col("datetime").dt.convert_time_zone("America/New_York").alias("datetime")
    )
    .sort(["ticker", "datetime"])
    .with_columns(
        pl.col("datetime").dt.date().alias("trading_day"),
        ((pl.col("high") + pl.col("low") + pl.col("close")) / 3).alias("price"),
    )
)

df_to_plot = add_indicators(
    df_to_plot,
    [
        daily_vwap,
        atr,
        bollinger_bands,
        rsi,
        donchian_channels,
        momentum,
    ],
)

df_to_plot

datetime,ticker,open,high,low,close,volume,trading_day,price,vwap_daily,atr_14,bb_middle_20,bb_upper_20,bb_lower_20,rsi_14,donchian_upper_20,donchian_lower_20,donchian_middle_20,momentum_10
"datetime[μs, America/New_York]",str,f64,f64,f64,f64,i64,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2026-08-07 04:00:00 EDT,"""AIXI""",1.08,1.134,1.01,1.01,5617,2026-08-07,1.051333,1.051333,null,null,null,null,null,null,null,null,null
2026-08-07 04:01:00 EDT,"""AIXI""",1.07,1.07,1.07,1.07,753,2026-08-07,1.07,1.05354,null,null,null,null,null,null,null,null,null
2026-08-07 04:02:00 EDT,"""AIXI""",1.05,1.05,1.05,1.05,142,2026-08-07,1.05,1.053463,null,null,null,null,null,null,null,null,null
2026-08-07 04:04:00 EDT,"""AIXI""",1.03,1.03,1.03,1.03,138,2026-08-07,1.03,1.052976,null,null,null,null,null,null,null,null,null
2026-08-07 04:06:00 EDT,"""AIXI""",1.0207,1.0207,1.0207,1.0207,974,2026-08-07,1.0207,1.048852,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-08-28 19:48:00 EDT,"""XTNT""",0.371,0.371,0.3666,0.3666,3638,2026-08-28,0.368067,0.402913,0.001921,0.366845,0.372453,0.361237,42.937978,0.3717,0.364,0.36785,0.0019
2026-08-28 19:49:00 EDT,"""XTNT""",0.3709,0.371,0.3709,0.3709,5540,2026-08-28,0.370933,0.40291,0.002186,0.36699,0.372854,0.361126,51.620203,0.3717,0.364,0.36785,0.0062
2026-08-28 19:57:00 EDT,"""XTNT""",0.3655,0.3699,0.3655,0.3699,703,2026-08-28,0.368433,0.40291,0.002529,0.367185,0.373162,0.361208,49.72534,0.3717,0.364,0.36785,0.0052


In [52]:
# import plotly.graph_objects as go
# from plotly.subplots import make_subplots


# GROUPS = ["ticker", "trading_day"]

# plot_df = (
#     df_to_plot
#     .sort(["ticker", "datetime"])
#     .with_row_index("plot_x")
# )


# def add_segmented_line(fig, frame, column, name, row, color, dash="solid", width=1):
#     """Añade una línea sin conectar distintos tickers ni días."""
#     show_legend = True

#     for segment in frame.partition_by(GROUPS, maintain_order=True):
#         valid_segment = segment.drop_nulls(column)

#         if valid_segment.height == 0:
#             continue

#         fig.add_trace(
#             go.Scatter(
#                 x=valid_segment["plot_x"].to_list(),
#                 y=valid_segment[column].to_list(),
#                 mode="lines",
#                 name=name,
#                 legendgroup=name,
#                 showlegend=show_legend,
#                 line={"color": color, "dash": dash, "width": width},
#             ),
#             row=row,
#             col=1,
#         )

#         show_legend = False


# volume_colors = [
#     "red" if close > open_price
#     else "green" if close < open_price
#     else "gray"
#     for open_price, close in zip(
#         plot_df["open"].to_list(),
#         plot_df["close"].to_list(),
#     )
# ]

# fig = make_subplots(
#     rows=5,
#     cols=1,
#     shared_xaxes=True,
#     vertical_spacing=0.025,
#     row_heights=[0.50, 0.15, 0.12, 0.12, 0.11],
#     subplot_titles=(
#         "Precio, VWAP, Bollinger y Donchian",
#         "Volumen",
#         "ATR 14",
#         "RSI 14",
#         "Momentum 10",
#     ),
# )

# fig.add_trace(
#     go.Candlestick(
#         x=plot_df["plot_x"].to_list(),
#         open=plot_df["open"].to_list(),
#         high=plot_df["high"].to_list(),
#         low=plot_df["low"].to_list(),
#         close=plot_df["close"].to_list(),
#         name="Precio",
#     ),
#     row=1,
#     col=1,
# )

# # Precio
# add_segmented_line(fig, plot_df, "vwap_daily", "VWAP", 1, "orange", width=2)

# # Bandas de Bollinger
# add_segmented_line(fig, plot_df, "bb_upper_20", "BB superior", 1, "#7f7f7f", "dot")
# add_segmented_line(fig, plot_df, "bb_middle_20", "BB media", 1, "#7f7f7f")
# add_segmented_line(fig, plot_df, "bb_lower_20", "BB inferior", 1, "#7f7f7f", "dot")

# # Canales de Donchian
# add_segmented_line(
#     fig, plot_df, "donchian_upper_20", "Donchian superior", 1, "#9467bd", "dash"
# )
# add_segmented_line(
#     fig, plot_df, "donchian_middle_20", "Donchian medio", 1, "#9467bd", "dot"
# )
# add_segmented_line(
#     fig, plot_df, "donchian_lower_20", "Donchian inferior", 1, "#9467bd", "dash"
# )

# # Volumen: vela alcista rojo, vela bajista verde
# fig.add_trace(
#     go.Bar(
#         x=plot_df["plot_x"].to_list(),
#         y=plot_df["volume"].to_list(),
#         marker_color=volume_colors,
#         name="Volumen",
#     ),
#     row=2,
#     col=1,
# )

# # Indicadores inferiores
# add_segmented_line(fig, plot_df, "atr_14", "ATR 14", 3, "#1f77b4", width=2)
# add_segmented_line(fig, plot_df, "rsi_14", "RSI 14", 4, "#ff7f0e", width=2)
# add_segmented_line(fig, plot_df, "momentum_10", "Momentum 10", 5, "#2ca02c", width=2)

# fig.add_hline(y=70, row=4, col=1, line_dash="dot", line_color="red")
# fig.add_hline(y=30, row=4, col=1, line_dash="dot", line_color="green")
# fig.update_yaxes(range=[0, 100], row=4, col=1)

# # Una marca de eje por inicio de cada día y ticker.
# day_starts = (
#     plot_df
#     .group_by(GROUPS, maintain_order=True)
#     .agg(pl.col("plot_x").min().alias("plot_x"))
#     .sort("plot_x")
# )

# fig.update_xaxes(
#     tickmode="array",
#     tickvals=day_starts["plot_x"].to_list(),
#     ticktext=[
#         f"{row['ticker']}<br>{row['trading_day']}"
#         for row in day_starts.to_dicts()
#     ],
#     row=5,
#     col=1,
# )

# # Etiqueta centrada sobre cada tramo de ticker.
# ticker_ranges = (
#     plot_df
#     .group_by("ticker", maintain_order=True)
#     .agg(
#         pl.col("plot_x").min().alias("start"),
#         pl.col("plot_x").max().alias("end"),
#     )
# )

# for ticker_range in ticker_ranges.to_dicts():
#     fig.add_annotation(
#         x=(ticker_range["start"] + ticker_range["end"]) / 2,
#         y=1.03,
#         xref="x",
#         yref="paper",
#         text=f"<b>{ticker_range['ticker']}</b>",
#         showarrow=False,
#     )

# fig.update_layout(
#     title="Precio e indicadores técnicos",
#     xaxis_rangeslider_visible=False,
#     height=1_300,
#     barmode="overlay",
# )

# fig.show()

Conversion de data para backtesting

In [53]:
def get_data_to_backtesting(df,ticker,date):
    data = df_to_plot.filter(pl.col('ticker')==ticker).filter(pl.col('datetime').dt.date()==date)
    data.columns = [c[0].upper()+c[1:].lower() for c in df_to_plot.columns]
    data = data.to_pandas().set_index('Datetime')
    return data    



In [54]:
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
import numpy as np
from backtesting.test import SMA, GOOG


class VwapCross(Strategy):


    max_bars = 5
    valor_R=2
    def bollinger_bands_indicator(self):
        return np.vstack(
            (
                self.data.Bb_upper_20,
                self.data.Bb_middle_20,
                self.data.Bb_lower_20,
            )
        )
    def donchian_channels_indicator(self):
            return np.vstack(
                (
                    self.data.Donchian_upper_20	,
                    self.data.Donchian_lower_20,
                    self.data.Donchian_middle_20,
                )
            )
    
    def vwap_indicator(self):
        return self.data.Vwap_daily
    def atr_indicator(self):
            return self.data.Atr_14
    def momentum_indicator(self):
            return self.data.Momentum_10
    def rsi_indicator(self):
         return self.data.Rsi_14
    def init(self):
        price = self.data.Close
        self.vwap = self.I(self.vwap_indicator)
        # self.atr = self.I(self.atr_indicator)
        self.momentum= self.I(self.momentum_indicator)
        self.rsi = self.I(self.rsi_indicator)

        # self.bb_upper, self.bb_middle, self.bb_lower = self.I(
        #     self.bollinger_bands_indicator,
        #     name=("BB superior", "BB media", "BB inferior"),
        #     overlay=True,
        # )
        # self.donchian_upper, self.donchian_middle, self.donchian_lower = self.I(
        #     self.donchian_channels_indicator,
        #     name=("donchian superior", "donchian media", "donchian inferior"),
        #     overlay=True,
        # )
        self.nbars=0
        # self.ma2 = self.I(SMA, price, 20)
        self.data_ant=None

    def next(self):
        # if crossover(self.data.Close, self.data.Vwap_daily) and not self.position.is_long:
        if self.data.Close[-1]> self.data.Vwap_daily[-1] and self.data.Close[-2]< self.data.Vwap_daily[-2] and not self.trades and not self.orders:
            recent_volumes = self.data.Volume[-4:-1]
            if len(recent_volumes) == 3 and (
                self.data.Volume[-1] * 30 >= sum(recent_volumes) * 11
            ):
                # sl= self.data.Low[-3:].min()
                # tp =self.data.Close+ (self.data.Close-sl)*2
                sl= self.data.Close[-1]-self.data.Atr_14[-1]*2
                sl2= self.data.Close[-1]-self.data.Atr_14[-1]*2
                sl3= self.data.Close[-1]-self.data.Atr_14[-1]*2
                tp =self.data.Close[-1]+self.data.Atr_14[-1]*2
                tp2 =self.data.Close[-1]+self.data.Atr_14[-1]*3
                tp3 =self.data.Close[-1]+self.data.Atr_14[-1]*6
                self.nbars=0
                if sl>0 and tp>0:
                    # print(self.data.Close[-1],sl,tp)

                    size = int(self.valor_R/(self.data.Close[-1]-sl))
                    size2=int (size/4)
                    size3=int (size/4)
                    size1 = size-size2-size3
                    # print(size)
                    # print(self.data.Close[-1],sl,tp,size,size1,size2,size3)
                    # self.buy(sl=sl,tp=tp,size=size1)
                    # self.buy(sl=sl2,tp=tp2,size=size2)
                    # self.buy(sl=sl3,tp=tp3,size=size3)
                    self.buy(sl=sl, tp=tp, size=size1, tag="salida 1")
                    self.buy(sl=sl2, size=size2, tag="salida 2")
                    self.buy(sl=sl3, size=size3, tag="salida 3")

        open_price = self.data.Open[-1]
        high_price = self.data.High[-1]
        low_price = self.data.Low[-1]
        close_price = self.data.Close[-1]
        previous_open = self.data.Open[-2]
        previous_close = self.data.Close[-2]
        
        body = abs(close_price - open_price)
        upper_wick = high_price - max(open_price, close_price)
        lower_wick = min(open_price, close_price) - low_price
        upper_wick_signal = (
            body > 0
            and upper_wick > body * 3
            and lower_wick < upper_wick*1.5
        )

        bearish_engulfing_signal = (
            previous_close > previous_open      # vela anterior alcista
            and close_price < open_price        # vela actual bajista
            and open_price >= previous_close    # abre por encima del cierre previo
            and close_price <= previous_open    # cierra por debajo de la apertura previa
            and body > abs(previous_close - previous_open)
        )

        salida_1_abierta = any(
            trade.tag == "salida 1" for trade in self.trades
        )

        if not salida_1_abierta:
            for trade in self.trades:
                if trade.tag == "salida 2" and upper_wick_signal:
                    trade.close()

                if trade.tag == "salida 3" and bearish_engulfing_signal:
                    trade.close()
                # elif crossover(self.data.Close, self.data.vwap_daily):
                    # self.sell()
                # if self.position.is_long:
                #     self.nbars+=1
                #     if self.nbars>=self.max_bars:
                        
                #         self.position.close()
                self.data_ant = self.data


In [55]:
from pathlib import Path
from datetime import timedelta
import math


def strategy_name(strategy):
    """Return the class name for a Backtesting.py strategy class or instance."""
    return strategy.__name__ if isinstance(strategy, type) else type(strategy).__name__


def build_backtest_output_dir(config, strategy, ticker, trading_date):
    """Build the output directory without creating it."""
    day = trading_date.date() if isinstance(trading_date, datetime) else trading_date
    return (
        Path(config["data_dir"])
        / "output"
        / strategy_name(strategy)
        / str(ticker)
        / day.strftime("%Y%m%d")
    )


def backtesting_stats_to_scalar_metrics(stats):
    """Keep only scalar Backtesting.py statistics suitable for Polars/Parquet."""
    metrics = {}
    for key, value in stats.items():
        if str(key).startswith("_"):
            continue

        if isinstance(value, np.generic):
            value = value.item()

        if isinstance(value, (datetime, date)):
            value = value.isoformat()
        elif isinstance(value, timedelta):
            value = value.total_seconds()
        elif isinstance(value, float) and not math.isfinite(value):
            value = None
        elif not isinstance(value, (str, int, float, bool, type(None))):
            continue

        metrics[str(key)] = value
    return metrics

In [56]:
def run_and_persist_backtest(ticker, trading_date):
    """Run and persist one ticker/date backtest using the current strategy settings."""
    day = trading_date.date() if isinstance(trading_date, datetime) else trading_date
    data = get_data_to_backtesting(df_to_plot, ticker, day)

    if data.empty:
        raise ValueError(f"No backtesting data available for {ticker} on {day:%Y-%m-%d}.")

    bt = Backtest(
        data,
        VwapCross,
        commission=(0.001, 0),
        cash=500,
        exclusive_orders=False,
    )
    stats = bt.run(valor_R=2)

    output_dir = build_backtest_output_dir(config, VwapCross, ticker, day)
    output_dir.mkdir(parents=True, exist_ok=True)

    result_row = {
        **backtesting_stats_to_scalar_metrics(stats),
        "strategy": strategy_name(VwapCross),
        "ticker": str(ticker),
        "date": day.isoformat(),
    }
    result = pl.DataFrame([result_row])
    result.write_parquet(output_dir / "stats.parquet")
    bt.plot(filename=str(output_dir / "backtest.html"), open_browser=False)
    return result


In [57]:
def run_and_persist_backtest_batch(df_runs):
    """Run, persist, and aggregate every ticker/date combination in df_runs."""
    results = []
    current_strategy = strategy_name(VwapCross)

    for run in df_runs.select("ticker", "date").iter_rows(named=True):
        ticker = str(run["ticker"])
        trading_date = run["date"]
        day = trading_date.date() if isinstance(trading_date, datetime) else trading_date

        try:
            result = run_and_persist_backtest(ticker, trading_date).with_columns(
                pl.lit("success").alias("status"),
                pl.lit(None, dtype=pl.Utf8).alias("error_message"),
            )
        except Exception as exc:
            result = pl.DataFrame(
                [
                    {
                        "strategy": current_strategy,
                        "ticker": ticker,
                        "date": day.isoformat(),
                        "status": "failed",
                        "error_message": str(exc),
                    }
                ]
            )

        results.append(result)

    if results:
        aggregated = pl.concat(results, how="diagonal_relaxed").sort(["ticker", "date"])
    else:
        aggregated = pl.DataFrame(
            schema={
                "strategy": pl.Utf8,
                "ticker": pl.Utf8,
                "date": pl.Utf8,
                "status": pl.Utf8,
                "error_message": pl.Utf8,
            }
        )

    output_dir = Path(config["data_dir"]) / "output" / current_strategy
    output_dir.mkdir(parents=True, exist_ok=True)
    aggregated.write_parquet(output_dir / "results.parquet")
    return aggregated


In [58]:
ticker = "VEEA"
trading_date = datetime.strptime("2026-10-09", "%Y-%m-%d")
backtest_result = run_and_persist_backtest(ticker, trading_date)
backtest_result


d:\00_PROG\23_algo_trd\021_sm_framework\venv\Lib\site-packages\bokeh\util\serialization.py:256: UserWarning:

no explicit representation of timezones available for np.datetime64



Start,End,Duration,Exposure Time [%],Equity Final [$],Equity Peak [$],Commissions [$],Return [%],Buy & Hold Return [%],Return (Ann.) [%],Volatility (Ann.) [%],CAGR [%],Sharpe Ratio,Sortino Ratio,Calmar Ratio,Alpha [%],Beta,Max. Drawdown [%],Avg. Drawdown [%],Max. Drawdown Duration,Avg. Drawdown Duration,# Trades,Win Rate [%],Best Trade [%],Worst Trade [%],Avg. Trade [%],Max. Trade Duration,Avg. Trade Duration,Profit Factor,Expectancy [%],SQN,Kelly Criterion,strategy,ticker,date
str,str,f64,f64,f64,f64,f64,f64,f64,null,null,f64,null,null,null,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str
"""2026-10-09T04:00:00-04:00""","""2026-10-09T19:59:00-04:00""",57540.0,19.30745,501.8309,502.2329,0.072,0.36618,-3.468208,null,null,642.336612,null,null,null,0.426381,0.017358,-1.542562,-0.568715,24180.0,13500.0,36,41.666667,12.261095,-3.07407,0.639087,4680.0,540.0,1.666916,0.709428,0.359883,0.057842,"""VwapCross""","""VEEA""","""2026-10-09"""


In [59]:
df_backtest_results = run_and_persist_backtest_batch(df_backtest_runs)
df_backtest_results

d:\00_PROG\23_algo_trd\021_sm_framework\venv\Lib\site-packages\bokeh\util\serialization.py:256: UserWarning:

no explicit representation of timezones available for np.datetime64

d:\00_PROG\23_algo_trd\021_sm_framework\venv\Lib\site-packages\bokeh\util\serialization.py:256: UserWarning:

no explicit representation of timezones available for np.datetime64

d:\00_PROG\23_algo_trd\021_sm_framework\venv\Lib\site-packages\bokeh\util\serialization.py:256: UserWarning:

no explicit representation of timezones available for np.datetime64

d:\00_PROG\23_algo_trd\021_sm_framework\venv\Lib\site-packages\bokeh\util\serialization.py:256: UserWarning:

no explicit representation of timezones available for np.datetime64

d:\00_PROG\23_algo_trd\021_sm_framework\venv\Lib\site-packages\bokeh\util\serialization.py:256: UserWarning:

no explicit representation of timezones available for np.datetime64

d:\00_PROG\23_algo_trd\021_sm_framework\venv\Lib\site-packages\bokeh\util\serialization.py:256: UserWarni

Start,End,Duration,Exposure Time [%],Equity Final [$],Equity Peak [$],Commissions [$],Return [%],Buy & Hold Return [%],Return (Ann.) [%],Volatility (Ann.) [%],CAGR [%],Sharpe Ratio,Sortino Ratio,Calmar Ratio,Alpha [%],Beta,Max. Drawdown [%],Avg. Drawdown [%],Max. Drawdown Duration,Avg. Drawdown Duration,# Trades,Win Rate [%],Best Trade [%],Worst Trade [%],Avg. Trade [%],Max. Trade Duration,Avg. Trade Duration,Profit Factor,Expectancy [%],SQN,Kelly Criterion,strategy,ticker,date,status,error_message
str,str,f64,f64,f64,f64,f64,f64,f64,null,null,f64,null,null,null,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str
"""2026-08-07T04:00:00-04:00""","""2026-08-07T19:59:00-04:00""",57540.0,4.528986,498.754186,501.3085,0.018,-0.249163,-0.943396,null,null,-74.544281,null,null,null,-0.245346,0.004045,-0.792107,-0.460551,31320.0,15720.0,9,33.333333,2.355503,-3.772515,-1.393138,540.0,420.0,0.282926,-1.367107,-0.574509,-0.188461,"""VwapCross""","""AIXI""","""2026-08-07""","""success""",null
"""2026-08-28T04:00:00-04:00""","""2026-08-28T19:59:00-04:00""",57540.0,9.80663,497.779371,511.041743,0.054,-0.444126,-5.6055,null,null,-91.294508,null,null,null,-0.127761,0.056438,-2.595164,-2.006996,33480.0,23160.0,27,14.814815,4.439174,-1.417286,-0.067942,1920.0,480.0,0.862361,-0.059219,-0.685518,-0.083458,"""VwapCross""","""AIXI""","""2026-08-28""","""success""",null
"""2026-09-04T09:30:00-04:00""","""2026-09-04T16:32:00-04:00""",25320.0,0.0,500.0,500.0,null,0.0,-7.782101,null,null,0.0,null,null,null,0.0,0.0,-0.0,null,null,null,0,null,null,null,null,null,null,null,null,null,null,"""VwapCross""","""APUS""","""2026-09-04""","""success""",null
"""2026-09-24T04:00:00-04:00""","""2026-09-24T19:59:00-04:00""",57540.0,4.460432,502.808314,505.300157,0.018,0.561663,164.220692,null,null,2058.072639,null,null,null,0.042007,0.003164,-0.493141,-0.30364,29100.0,20460.0,9,44.444444,71.557093,-5.752765,9.441156,780.0,360.0,6.027487,11.650222,0.933198,0.24958,"""VwapCross""","""APUS""","""2026-09-24""","""success""",null
null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""VwapCross""","""APUS""","""2026-10-07""","""failed""","""size must be a positive fracti…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""2026-10-06T04:00:00-04:00""","""2026-10-06T19:59:00-04:00""",57540.0,24.810127,499.741614,504.321543,0.042,-0.051677,35.033365,null,null,-24.685284,null,null,null,-0.573889,0.014906,-1.10474,-0.443977,35940.0,9360.0,21,42.857143,20.996019,-9.145634,0.572712,8640.0,1440.0,1.366991,0.799373,-0.078573,-0.017215,"""VwapCross""","""XHLD""","""2026-10-06""","""success""",null
"""2026-08-04T07:16:00-04:00""","""2026-08-04T16:05:00-04:00""",31740.0,3.571429,503.203214,503.203214,0.002,0.640643,5.640054,null,null,57108.998211,null,null,null,0.349161,0.051681,-0.373003,-0.373003,3720.0,3720.0,1,100.0,1.82488,1.82488,1.82488,0.0,0.0,null,1.82488,null,null,"""VwapCross""","""XPON""","""2026-08-04""","""success""",null
null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""VwapCross""","""XPON""","""2026-08-24""","""failed""","""size must be a positive fracti…"


In [60]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

kpi_columns = [
    "Return [%]",
    "Buy & Hold Return [%]",
    "Win Rate [%]",
    "Avg. Trade [%]",
    "Avg. Drawdown [%]",
    "Expectancy [%]",
]

missing_kpi_columns = [column for column in kpi_columns if column not in df_backtest_results.columns]
if missing_kpi_columns:
    raise ValueError(
        "Cannot plot KPI histograms because df_backtest_results is missing: "
        + ", ".join(missing_kpi_columns)
    )

successful_backtests = df_backtest_results.filter(pl.col("status") == "success")
if successful_backtests.height == 0:
    raise ValueError("Cannot plot KPI histograms because there are no successful backtest results.")

fig = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=kpi_columns,
)

for index, kpi in enumerate(kpi_columns):
    row = index // 3 + 1
    col = index % 3 + 1
    values = successful_backtests.get_column(kpi).drop_nulls().to_list()
    fig.add_trace(
        go.Histogram(x=values, name=kpi, showlegend=False),
        row=row,
        col=col,
    )
    fig.update_xaxes(title_text=kpi, row=row, col=col)
    fig.update_yaxes(title_text="Count", row=row, col=col)

fig.update_layout(
    title="Backtest KPI distributions",
    height=700,
    width=1_300,
)
fig.show()
